# UdaciHeadline: LLM Inference Optimization Project

## Project Introduction
Large Language Models (LLMs) are transforming content creation, but deploying them efficiently remains a major hurdle. Imagine you're an ML Engineer at a bustling online news portal. Your key task? Automatically generating catchy headlines from article summaries using an LLM. The problem? The current inference process is sluggish, causing publication delays and driving up operational costs. In this project, UdaciHeadline, you'll step into this role and tackle this critical challenge head-on. Your mission is to accelerate the headline generation pipeline significantly by applying state-of-the-art LLM inference optimization techniques. Get ready to dive deep into practical optimization and deployment!

## Project Summary
This project provides hands-on experience in optimizing the inference performance of a pre-trained Large Language Model (like Llama-3.2-1B) for news headline generation. You will bring together concepts of LLM architecture, optimization techniques, and deployment frameworks. Specifically, you will:

1.  **Establish a baseline** inference pipeline and profile its performance.
2.  Implement and evaluate architectural optimizations like **KV-caching**.
3.  Apply model compression techniques like **quantization** and **pruning**.
4.  Configure and benchmark **distributed inference** using Tensor and Pipeline Parallelism.
5.  Apply advanced decoding mechanisms like **speculative decoding**.
6.  Perform comprehensive **benchmarking and analysis** across all stages.
7.  Produce a **final report** summarizing findings and trade-offs.

## Imports and Global Configuration

Let's import the libraries we'll use throughout the project and define some constants like the model name and the prompt template.

In [1]:
import os
import sys
import gc
import json
import math
import time
import platform
import logging
import warnings
import subprocess
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import psutil
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load as load_metric
from pprint import pprint
import torch.nn.utils.prune as prune

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# ---- Constants ----
# Model resolution: the Udacity workspace ships Llama-3.2-1B locally; elsewhere use the ungated Hub mirror
# of the same weights (override with UDACI_MODEL / UDACI_TARGET_MODEL).
_LOCAL_1B = "/voc/shared/models/llama/Llama-3.2-1B"
_LOCAL_3B = "/voc/shared/models/llama/Llama-3.2-3B"
MODEL_NAME = os.environ.get("UDACI_MODEL", _LOCAL_1B if os.path.isdir(_LOCAL_1B) else "unsloth/Llama-3.2-1B")
TARGET_MODEL_NAME = os.environ.get("UDACI_TARGET_MODEL", _LOCAL_3B if os.path.isdir(_LOCAL_3B) else "unsloth/Llama-3.2-3B")
if os.path.isdir(MODEL_NAME):
    os.environ["HF_HUB_OFFLINE"] = "1"      # only go offline when everything is available locally

DATASET_PATH = os.environ.get("UDACI_DATASET", "../dataset/News_Category_Dataset.json")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 on GPU (Llama checkpoints are stored in bf16); fp32 on CPU (fastest CPU kernels, no bf16 emulation)
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

MAX_NEW_TOKENS = 24            # Max length for the generated headline (HuffPost headlines are ~10-15 tokens)
N_EVAL = int(os.environ.get("UDACI_N_EVAL", 20))   # evaluation samples per configuration
N_SHOT = 2                     # few-shot examples in the prompt (base model, not instruction-tuned)
SEED = 42
FORCE_RERUN = os.environ.get("UDACI_FORCE_RERUN", "0") == "1"   # ignore cached results/*.json

PROMPT = \
"""You are a news editor. Write a short, catchy headline for each article summary.

{examples}Summary: {summary}
Headline:"""

torch.manual_seed(SEED)
print(f"Device: {DEVICE} | dtype: {DTYPE} | model: {MODEL_NAME} | target (spec. decoding): {TARGET_MODEL_NAME}")
print(f"N_EVAL={N_EVAL}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}, N_SHOT={N_SHOT}, results -> {RESULTS_DIR.resolve()}")


Device: cpu | dtype: torch.float32 | model: unsloth/Llama-3.2-1B | target (spec. decoding): unsloth/Llama-3.2-3B
N_EVAL=20, MAX_NEW_TOKENS=24, N_SHOT=2, results -> /home/architect/UdaciHeadline/project/results


In [2]:
def environment_info():
    """Hardware/software details recorded for reproducibility."""
    import transformers, datasets, evaluate, accelerate
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "evaluate": evaluate.__version__,
        "accelerate": accelerate.__version__,
        "cpu": platform.processor() or "unknown",
        "cpu_count": os.cpu_count(),
        "torch_threads": torch.get_num_threads(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "cuda_available": torch.cuda.is_available(),
        "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "device": DEVICE, "dtype": str(DTYPE), "model": MODEL_NAME, "target_model": TARGET_MODEL_NAME,
        "n_eval": N_EVAL, "max_new_tokens": MAX_NEW_TOKENS,
    }
    try:
        info["bitsandbytes"] = __import__("bitsandbytes").__version__
    except Exception:
        info["bitsandbytes"] = None
    try:  # nicer CPU name on Linux
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu"] = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    return info

ENV = environment_info()
json.dump(ENV, open(RESULTS_DIR / "environment.json", "w"), indent=2)
pprint(ENV)


{'accelerate': '1.14.0',
 'bitsandbytes': '0.50.1',
 'cpu': 'Intel(R) Core(TM) i7-10610U CPU @ 1.80GHz',
 'cpu_count': 8,
 'cuda_available': False,
 'datasets': '5.0.1',
 'device': 'cpu',
 'dtype': 'torch.float32',
 'evaluate': '0.4.6',
 'gpus': [],
 'max_new_tokens': 24,
 'model': 'unsloth/Llama-3.2-1B',
 'n_eval': 20,
 'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39',
 'python': '3.12.3',
 'ram_total_gb': 33.3,
 'target_model': 'unsloth/Llama-3.2-3B',
 'torch': '2.13.0+cpu',
 'torch_threads': 4,
 'transformers': '5.15.0'}


## Data Loading

We will use the "News Category Dataset" from Kaggle. The `kagglehub` library makes it easy to download and access. Your task is to implement the function to load and preprocess the data according to the docstring.

In [3]:
def load_news_dataset(path, n_eval=N_EVAL, n_shot=N_SHOT, seed=SEED):
    """Load the News Category Dataset (JSON lines) with the HF `datasets` library and prepare it for
    headline generation.

    Steps
    1. `load_dataset("json")` -> ~210k HuffPost articles.
    2. Keep only articles that have both a headline and a reasonably informative short_description
       (>= 15 words) so the model has something to summarise, and drop very long ones (> 80 words) to keep
       prompts short.
    3. Shuffle with a fixed seed and split off `n_shot` few-shot examples (used inside the prompt) and
       `n_eval` evaluation samples.  The same samples are used for every configuration.
    Returns (eval_dataset, fewshot_examples) where the dataset has columns summary / headline / category.
    """
    raw = load_dataset("json", data_files=path, split="train")

    def _ok(ex):
        s, h = ex.get("short_description") or "", ex.get("headline") or ""
        n = len(s.split())
        return 15 <= n <= 80 and len(h.split()) >= 4

    ds = raw.filter(_ok)
    ds = ds.rename_column("short_description", "summary").select_columns(["summary", "headline", "category"])
    ds = ds.shuffle(seed=seed)
    fewshot = [ds[i] for i in range(n_shot)]
    eval_ds = ds.select(range(n_shot, n_shot + n_eval))
    print(f"Loaded {len(raw):,} articles, {len(ds):,} after filtering; using {n_shot} few-shot + {len(eval_ds)} eval samples.")
    return eval_ds, fewshot


eval_dataset, FEWSHOT = load_news_dataset(DATASET_PATH)
FEWSHOT_BLOCK = "".join(f"Summary: {ex['summary']}\nHeadline: {ex['headline']}\n\n" for ex in FEWSHOT)

def build_prompt(summary):
    return PROMPT.format(examples=FEWSHOT_BLOCK, summary=summary.strip())

print(build_prompt(eval_dataset[0]["summary"]))
print("\nReference headline:", eval_dataset[0]["headline"])


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/209527 [00:00<?, ? examples/s]

Loaded 209,527 articles, 127,270 after filtering; using 2 few-shot + 20 eval samples.
You are a news editor. Write a short, catchy headline for each article summary.

Summary: An end-of-life discussion is not a conversation likely to arise spontaneously on its own. Whether you are an aging parent or a concerned adult child, you must make the first move. Seize any opportunity to begin the conversation.
Headline: Hospice: Having an End-of-Life Conversation in the Midst of Life, Part 1

Summary: Some online deals are too easy to find and too hard to pass up. My new fondness for coupons started when I was visiting my folks in Florida. Before entering the Gap Outlet somewhere along the Gulf coast, I found an Internet coupon on the sidewalk: 50 percent off any purchase!
Headline: Learning From My Elders: How to Use Online Coupons

Summary: FYI, Americans: The BAFTAs are tomorrow, the same night as the Grammys. Watson, of course, is the face of Lancome; her most
Headline:

Reference headline:

# 2. Baseline Performance

Before we can optimize, we need a starting point. Here, you'll establish the baseline performance of the `Llama-3.2-1B` model without any specific optimizations. We will measure latency, throughput, and the quality of the generated headlines using the ROUGE score.

### Your Task: Implement the Evaluation Pipeline
You need to implement the core functions for loading a model, generating a headline, and evaluating performance. These functions will be reused for every optimization technique.

In [4]:
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def _peak_memory_gb():
    """Peak memory during the last measurement window: CUDA allocator peak on GPU, process RSS on CPU."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1e9
    return psutil.Process().memory_info().rss / 1e9

def model_footprint_gb(model):
    try:
        return model.get_memory_footprint() / 1e9
    except Exception:
        return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9


def load_model(model_name, quantization_config=None, device_map=None, dtype=None):
    """Load a tokenizer + causal LM.

    * `dtype` defaults to DTYPE (bf16 on GPU, fp32 on CPU).
    * `quantization_config` (BitsAndBytesConfig) enables 8/4-bit loading; bitsandbytes needs a device_map,
      so one is supplied automatically ("auto" on GPU, everything on CPU otherwise).
    * `device_map` ("auto", "balanced", or an explicit dict) hands placement to `accelerate` -- this is how
      tensor / pipeline parallel sharding across several GPUs is requested. Without it the model is moved to
      DEVICE as a whole.
    Returns (model, tokenizer). Padding side is set to left so batched generation works.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    kwargs = {"dtype": dtype or DTYPE, "low_cpu_mem_usage": True}
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config
        if device_map is None:
            device_map = "auto" if DEVICE == "cuda" else {"": "cpu"}
    if device_map is not None:
        kwargs["device_map"] = device_map

    t0 = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    if device_map is None:
        model.to(DEVICE)
    model.eval()
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print(f"Loaded {model_name} in {time.perf_counter()-t0:.1f}s | footprint {model_footprint_gb(model):.2f} GB"
          f" | dtype {next(model.parameters()).dtype} | device_map: {getattr(model, 'hf_device_map', None) or DEVICE}")
    return model, tokenizer


def _clean_headline(text):
    """First non-empty line of the continuation, without quotes/trailing junk."""
    for line in text.split("\n"):
        line = line.strip().strip('"').strip()
        if line:
            return line
    return text.strip()


def generate_headline(model, tokenizer, summary, generation_args):
    """Generate one headline and measure its latency.

    Returns (headline:str, latency_s:float, new_tokens:int). Timing brackets only `generate()` (tokenisation
    excluded) and is synchronised on CUDA. Generation stops at MAX_NEW_TOKENS, EOS or the first newline.
    """
    prompt = build_prompt(summary)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    args = dict(generation_args)
    args.setdefault("max_new_tokens", MAX_NEW_TOKENS)
    args.setdefault("do_sample", False)
    args.setdefault("pad_token_id", tokenizer.pad_token_id)
    # stop at the end of the headline line
    args.setdefault("stop_strings", ["\n"])
    args.setdefault("tokenizer", tokenizer)

    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, **args)
    _sync()
    latency = time.perf_counter() - t0

    new_ids = out[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    n_new = int((new_ids != tokenizer.pad_token_id).sum().item()) or new_ids.numel()
    return _clean_headline(text), latency, n_new


ROUGE = load_metric("rouge")

def report_metrics(results, latencies, max_new_tokens, label="", extra=None, verbose=True):
    """Compute and print latency / throughput / memory / ROUGE metrics.

    results   : list of dicts with keys headline, reference, new_tokens
    latencies : per-sample generate() latencies in seconds
    """
    lat = np.asarray(latencies, dtype=float)
    toks = np.asarray([r["new_tokens"] for r in results], dtype=float)
    rouge = ROUGE.compute(predictions=[r["headline"] for r in results],
                          references=[r["reference"] for r in results], use_stemmer=True)
    m = {
        "label": label,
        "n_samples": len(results),
        "max_new_tokens": max_new_tokens,
        "latency_mean_s": float(lat.mean()),
        "latency_std_s": float(lat.std()),
        "latency_p50_s": float(np.percentile(lat, 50)),
        "latency_p99_s": float(np.percentile(lat, 99)),
        "latency_min_s": float(lat.min()),
        "latency_max_s": float(lat.max()),
        "total_time_s": float(lat.sum()),
        "avg_new_tokens": float(toks.mean()),
        "throughput_tok_s": float(toks.sum() / lat.sum()),          # generated tokens per second
        "throughput_samples_s": float(len(results) / lat.sum()),    # headlines per second
        "rouge1": float(rouge["rouge1"]), "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]), "rougeLsum": float(rouge["rougeLsum"]),
    }
    if extra:
        m.update(extra)
    if verbose:
        print(f"\n=== {label or 'metrics'} ===")
        print(f"  samples            : {m['n_samples']}  (avg {m['avg_new_tokens']:.1f} new tokens, cap {max_new_tokens})")
        print(f"  latency mean / p50 / p99 : {m['latency_mean_s']:.3f} / {m['latency_p50_s']:.3f} / {m['latency_p99_s']:.3f} s")
        print(f"  throughput         : {m['throughput_tok_s']:.2f} tokens/s  ({m['throughput_samples_s']*60:.1f} headlines/min)")
        if "peak_memory_gb" in m:
            print(f"  memory             : model {m.get('model_footprint_gb', float('nan')):.2f} GB, peak {m['peak_memory_gb']:.2f} GB ({m.get('memory_kind','')})")
        print(f"  ROUGE-1 / -2 / -L  : {m['rouge1']:.4f} / {m['rouge2']:.4f} / {m['rougeL']:.4f}")
    return m


def evaluate_model(dataset, model, tokenizer, generation_args, n=N_EVAL, label="run", extra=None,
                   warmup=True, save=True, verbose=True, cache=True):
    """Run headline generation over the first `n` samples and report metrics.

    * one warm-up generation (kernel/JIT/page-in) that is not timed
    * peak-memory stats reset before the loop and read after it
    * results are stored under results/<label>.json (and re-loaded on later runs unless FORCE_RERUN=1)
    """
    path = RESULTS_DIR / f"{label}.json"
    if cache and not FORCE_RERUN and path.exists():
        m = json.load(open(path))
        print(f"[cache] loaded {path}  (set UDACI_FORCE_RERUN=1 to recompute)")
        if verbose:
            report_metrics(m["samples"], [s["latency_s"] for s in m["samples"]], m["max_new_tokens"], label=label,
                           extra={k: v for k, v in m.items() if k not in ("samples",)}, verbose=True)
        return m

    n = min(n, len(dataset))
    if warmup:
        generate_headline(model, tokenizer, dataset[0]["summary"], generation_args)
    gc.collect(); _reset_peak_memory()
    rss_before = psutil.Process().memory_info().rss / 1e9

    results, latencies = [], []
    t_start = time.perf_counter()
    for i in range(n):
        ex = dataset[i]
        headline, lat, n_new = generate_headline(model, tokenizer, ex["summary"], generation_args)
        results.append({"headline": headline, "reference": ex["headline"], "new_tokens": n_new, "latency_s": lat})
        latencies.append(lat)
        if verbose and (i < 3 or i == n - 1):
            print(f"  [{i+1:>2}/{n}] {lat:6.2f}s {n_new:>2} tok | gen: {headline[:70]!r}\n{'':14}| ref: {ex['headline'][:70]!r}")
    wall = time.perf_counter() - t_start

    mem_extra = {
        "model_footprint_gb": model_footprint_gb(model),
        "peak_memory_gb": _peak_memory_gb(),
        "memory_kind": "cuda max_memory_allocated" if DEVICE == "cuda" else "process RSS",
        "rss_delta_gb": psutil.Process().memory_info().rss / 1e9 - rss_before,
        "wall_time_s": wall,
        "generation_args": {k: (str(v) if not isinstance(v, (int, float, bool, str, type(None))) else v)
                            for k, v in generation_args.items() if k != "tokenizer"},
        "device": DEVICE, "dtype": str(next(model.parameters()).dtype),
    }
    if extra:
        mem_extra.update(extra)
    m = report_metrics(results, latencies, generation_args.get("max_new_tokens", MAX_NEW_TOKENS), label=label,
                       extra=mem_extra, verbose=verbose)
    m["samples"] = results
    if save:
        json.dump(m, open(path, "w"), indent=2)
        print(f"  saved -> {path}")
    return m


def show_headlines(metrics, k=5):
    """Pretty-print k generated vs reference headlines."""
    rows = [{"generated": s["headline"], "reference": s["reference"], "tokens": s["new_tokens"],
             "latency_s": round(s["latency_s"], 2)} for s in metrics["samples"][:k]]
    with pd.option_context("display.max_colwidth", 90, "display.width", 200):
        display(pd.DataFrame(rows))


In [5]:
# ---- Baseline: plain autoregressive decoding WITHOUT the KV cache ----
# Every decoding step re-computes attention over the whole prefix, i.e. O(n^2) work per headline.
model, tokenizer = load_model(MODEL_NAME)

baseline_args = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "use_cache": False}
baseline_metrics = evaluate_model(eval_dataset, model, tokenizer, baseline_args, n=N_EVAL, label="baseline_no_cache")
show_headlines(baseline_metrics)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded unsloth/Llama-3.2-1B in 2.2s | footprint 4.94 GB | dtype torch.float32 | device_map: cpu


  [ 1/20]  81.15s 16 tok | gen: 'The BAFTAs: A Night of Glamour and Glamourous Stars'
              | ref: "Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS)"


  [ 2/20]  45.84s  9 tok | gen: 'The End of Life: A Personal Journey'
              | ref: 'The Conversation Nobody Wants To Have -- But Should'


  [ 3/20]  39.47s  9 tok | gen: 'The Inking of the Inkblot'
              | ref: 'Republicans Sprinting Toward Tax Cuts, Deficits Be Damned'


  [20/20] 110.62s 12 tok | gen: "Indiana's Religious Freedom Law: A Threat to the Constitution"
              | ref: 'Indiana Takes on America: Discrimination Against Gays, Religious Freed'



=== baseline_no_cache ===
  samples            : 20  (avg 11.9 new tokens, cap 24)
  latency mean / p50 / p99 : 58.388 / 54.896 / 109.534 s
  throughput         : 0.20 tokens/s  (1.0 headlines/min)
  memory             : model 4.94 GB, peak 6.20 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1539 / 0.0276 / 0.1426
  saved -> results/baseline_no_cache.json


,generated,reference,tokens,latency_s
0,The BAFTAs: A Night of Glamour and Glamourous Stars,Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS),16,81.15
1,The End of Life: A Personal Journey,The Conversation Nobody Wants To Have -- But Should,9,45.84
2,The Inking of the Inkblot,"Republicans Sprinting Toward Tax Cuts, Deficits Be Damned",9,39.47
3,Heart Disease: A Reminder to Be Wary of Symptoms,A New Dimension to Christmas Leads to Year-Round Devotion,12,55.37
4,Antibiotic Use in Hospitals: A Call for Appropriate Use,"1 In 25 Patients Experience Infection Related To Hospital Stay, Report Shows",13,54.42


In [6]:
# ---- Optional deeper look: PyTorch profiler on one baseline generation (top operators) ----
import torch.profiler
_ex = eval_dataset[0]["summary"]
_acts = [torch.profiler.ProfilerActivity.CPU] + ([torch.profiler.ProfilerActivity.CUDA] if DEVICE == "cuda" else [])
with torch.profiler.profile(activities=_acts, profile_memory=True) as prof:
    with torch.profiler.record_function("baseline_generate"):
        generate_headline(model, tokenizer, _ex, baseline_args)
sort_key = "self_cuda_time_total" if DEVICE == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=8))
prof.export_chrome_trace(str(RESULTS_DIR / "baseline_trace.json"))


USDT:2026-08-16 16:25:22 319673:319673 SyncActivityProfilerHandler.cpp:52] profiler_start


[W816 16:25:25.160024188 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


USDT:2026-08-16 16:27:04 319673:319673 SyncActivityProfilerHandler.cpp:59] profiler_stop


-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             aten::mm        91.99%       93.604s        91.99%       93.607s      51.774ms       4.74 GB       4.74 GB          1808  
                                    baseline_generate         2.90%        2.947s       100.00%      101.758s      101.758s      16.63 MB     -14.15 GB             1  
    aten::_scaled_dot_product_flash_attention_for_cpu         2.00%        2.039s         2.02%        2.057s       8.035ms     427.58 MB     -69.12 MB         

# 3. Architectural Optimization: KV Caching

**Your Task:** One of the most effective ways to speed up token generation is using a Key-Value (KV) cache. This avoids re-computing attention scores for tokens that are already part of the sequence. Enable the `use_cache` flag in the generation arguments and re-run the evaluation. Observe the impact on latency and throughput.

In [7]:
# TODO: Evaluate the model with KV Caching enabled.

# 4. Model Compression: Pruning

**Your Task:** Pruning removes redundant model weights, which can reduce model size and potentially speed up inference. Here, you will implement unstructured, magnitude-based pruning by creating a function that applies it to the model's linear layers and then evaluating the result.

In [8]:
def prune_model_weights(model, amount=0.3):
    """TODO: Applies L1 unstructured pruning to the linear layers of a model."""
    pass

# TODO: Evaluate the pruned model.

# 5. Model Compression: Quantization

**Your Task:** Quantization reduces the precision of model weights (e.g., from 16-bit to 4-bit), significantly cutting down memory usage and often speeding up inference. You will define a 4-bit quantization configuration and use it to load and evaluate a new model.

In [9]:
# TODO: Implement and evaluate 4-bit quantization.

# 6. Distributed Inference (Multi-GPU)

**Your Task:** If you have multiple GPUs, you can split the model across them to reduce the memory burden on a single GPU and potentially improve latency. We will explore two common techniques: Tensor Parallelism and Pipeline Parallelism.

*Note: This section requires a multi-GPU environment.*

### Tensor Parallelism
Tensor parallelism splits individual model layers (the tensors) across multiple GPUs. Operations like matrix multiplications are executed in parallel on different GPUs, and the results are aggregated. This is highly effective for reducing the memory footprint of very large layers. The `accelerate` library can handle this automatically via `device_map="auto"`.

### Pipeline Parallelism
Pipeline parallelism assigns entire layers or blocks of layers to different GPUs, creating a sequence or "pipeline" that the data flows through. For example, layers 1-10 run on GPU 0, layers 11-20 run on GPU 1, and so on. This is useful for very deep models where even a single layer might be too large for one GPU after tensor parallelism.

In [10]:
# TODO: Check for multi-GPU environment and evaluate with Tensor Parallelism.
# The `device_map="auto"` in your `load_model` function should automatically apply this.

In [11]:
# TODO: Evaluate with Pipeline Parallelism.
# This is more advanced and may require manually defining a device_map to assign
# different layers of the model to different GPUs.

# 7. Advanced Decoding: Speculative Decoding

**Your Task:** Speculative decoding uses a smaller, faster "draft" model to generate several candidate tokens. A larger, more accurate "target" model then verifies these tokens in a single forward pass. This can significantly speed up generation if the draft model is a good predictor. You will load a larger target model and a smaller draft model, benchmark the target model alone, and then benchmark it with assistance from the draft model.

In [12]:
# TODO: Implement and evaluate speculative decoding.

# 8. Final Report and Analysis

**Your Task:** Consolidate your findings into a summary report. 

1.  Fill in the Markdown table below with the **Latency**, **Throughput**, and **ROUGE scores** for each optimization technique you implemented.
2. Compile the final Project Report in PDF format:
    *   Document the entire process, detailing the methodology, techniques, and libraries used.
    *   Present the final benchmark results clearly.
    *   Provide a thorough analysis of the trade-offs between performance, resources, and quality for each optimization step.
    *   Conclude with recommendations for the most effective optimization strategy for this specific headline generation task, supported by your data.

Some example questions for discussing the trade-offs:
    *   Which method gave the best performance improvement?
    *   Did any methods significantly hurt the ROUGE score (quality)?
    *   Which optimization would you recommend for deployment in a production environment at the news portal, and why? Consider factors like cost, complexity, and performance.

## Performance Comparison

| Optimization Technique | Mean Latency (s) | Throughput (tokens/s) | ROUGE-1 Score |
|--------------------------|------------------|-----------------------|---------------|
| Baseline (No Cache)      | TODO             | TODO                  | TODO          |
| KV Caching               | TODO             | TODO                  | TODO          |
| Pruning (30%)            | TODO             | TODO                  | TODO          |
| Quantization (4-bit)     | TODO             | TODO                  | TODO          |
| Tensor Parallelism       | TODO             | TODO                  | TODO          |
| Pipeline Parallelism     | TODO             | TODO                  | TODO          |
| Speculative Decoding     | TODO             | TODO                  | TODO          |

---

